# Atlas - agente para o benchmark GAIA

Esse é o projeto final que fiz no curso de agentes de IA da Hugging Face (AI Agents Course).

A ideia é criar um agente que responde perguntas do benchmark GAIA. São perguntas fáceis para uma pessoa, mas difíceis para um modelo de linguagem sozinho, porque precisam de pesquisa na internet, leitura de arquivos e alguns passos de raciocínio.

Para o certificado do curso, o agente precisava acertar pelo menos 30% de 20 perguntas.

O que eu usei:
- smolagents (framework de agentes da Hugging Face)
- GPT-4o da OpenAI como modelo
- Whisper para transcrever áudio

## Como o agente funciona

O agente segue um ciclo simples:

1. **Pensa** no que precisa fazer
2. **Age**: escreve um código Python que chama uma ferramenta (por exemplo, uma busca no Google)
3. **Observa** o resultado
4. Repete até achar a resposta

Usei o `CodeAgent` do smolagents. A diferença dele é que o modelo escreve código Python em vez de só chamar uma ferramenta por vez. Isso ajuda bastante quando a pergunta envolve contas ou planilhas.

As ferramentas que o agente tem:
- busca na web (DuckDuckGo)
- abrir páginas da internet
- busca na Wikipedia
- transcrever áudio
- analisar imagens

## Instalação

In [ ]:
%pip install -q "smolagents>=1.20" openai ddgs wikipedia-api markdownify pandas openpyxl huggingface_hub requests

## Chaves de API

Preciso de duas chaves:
- `OPENAI_API_KEY`: para usar o GPT-4o e o Whisper
- `HF_TOKEN` (opcional): para baixar os arquivos das perguntas direto do dataset do GAIA no Hugging Face

As chaves são digitadas quando o notebook roda, assim elas não ficam salvas no arquivo.

In [ ]:
import os
from getpass import getpass

for key in ["OPENAI_API_KEY", "HF_TOKEN"]:
    if not os.getenv(key):
        value = getpass(f"{key} (Enter para pular): ")
        if value:
            os.environ[key] = value

print("OPENAI_API_KEY:", "ok" if os.getenv("OPENAI_API_KEY") else "FALTANDO")
print("HF_TOKEN:", "ok" if os.getenv("HF_TOKEN") else "não definido (opcional)")

## Imports

In [ ]:
import os
import re
import base64
import tempfile

import requests
import pandas as pd
from markdownify import markdownify
from openai import OpenAI
from huggingface_hub import hf_hub_download
from smolagents import (
    CodeAgent,
    OpenAIServerModel,
    DuckDuckGoSearchTool,
    WikipediaSearchTool,
    tool,
)

API_URL = "https://agents-course-unit4-scoring.hf.space"  # API de avaliação do curso
MODEL_ID = "gpt-4o"
MAX_STEPS = 12

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

## Ferramentas

No smolagents, para criar uma ferramenta basta escrever uma função com `@tool` em cima e uma docstring explicando o que ela faz. O modelo lê essa docstring para saber quando usar a ferramenta.

Criei três ferramentas minhas:
- `visit_webpage`: abre uma página e devolve o texto. Fiz a minha porque a Wikipedia estava bloqueando a ferramenta padrão.
- `transcribe_audio`: transcreve arquivos de áudio usando o Whisper.
- `analyze_image`: manda a imagem para o GPT-4o e faz uma pergunta sobre ela.

In [ ]:
@tool
def visit_webpage(url: str) -> str:
    """Opens a webpage and returns its content as markdown text.

    Args:
        url: The full URL of the page to open.
    """
    headers = {"User-Agent": "Mozilla/5.0 (compatible; Atlas-GAIA-agent/1.0)"}
    r = requests.get(url, headers=headers, timeout=20)
    r.raise_for_status()
    return markdownify(r.text)[:40000]


@tool
def transcribe_audio(file_path: str) -> str:
    """Transcribes an audio file (mp3, wav, m4a) to text.

    Args:
        file_path: Local path to the audio file.
    """
    with open(file_path, "rb") as f:
        result = client.audio.transcriptions.create(model="whisper-1", file=f)
    return result.text


@tool
def analyze_image(file_path: str, question: str) -> str:
    """Looks at an image file and answers a question about it.

    Args:
        file_path: Local path to the image (png, jpg).
        question: What you want to know about the image.
    """
    ext = file_path.split(".")[-1].lower()
    mime = "jpeg" if ext in ("jpg", "jpeg") else ext
    with open(file_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    resp = client.chat.completions.create(
        model=MODEL_ID,
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": question},
                {"type": "image_url", "image_url": {"url": f"data:image/{mime};base64,{b64}"}},
            ],
        }],
    )
    return resp.choices[0].message.content

## Prompt

A correção é feita comparando a resposta exatamente com o gabarito. Se a resposta certa é `3` e o agente responde `3 albums`, conta como erro.

Por isso o prompt explica bem o formato da resposta: só o número, sem unidade, listas separadas por vírgula, etc.

In [ ]:
INSTRUCTIONS = """You are a general AI assistant solving a benchmark question.
Think step by step and use your tools (web search, visit webpages, Wikipedia, Python,
audio transcription, image analysis) to find the correct answer. Verify facts.

When you call final_answer, give ONLY the answer, nothing else:
- A number: digits only, no commas as thousands separators, no units ($, %) unless asked.
- A string: as few words as possible, no articles, no abbreviations.
- A list: a single comma-separated string, following the rules above for each element.
- Follow any format the question asks for exactly (e.g. alphabetical order, plural form).
- Never write 'FINAL ANSWER' or explanations.
- Never invent or simulate data, files or transcriptions. If a file path is given, use exactly that path.
- If a webpage is blocked, try another source instead of guessing.
- In each Thought, check whether the last Observation actually answers the question before calling final_answer."""

## O agente

A classe `AtlasAgent` faz três coisas:

1. Cria o `CodeAgent` com o modelo e as ferramentas
2. Baixa o arquivo da pergunta, quando tem um (planilha, áudio, imagem). Se a API do curso falhar, tenta baixar do dataset do GAIA
3. Limpa a resposta final. Por exemplo, se vier uma lista `['a', 'b']`, transforma em `a, b`

In [ ]:
class AtlasAgent:
    def __init__(self, model_id: str = MODEL_ID, max_steps: int = MAX_STEPS):
        model = OpenAIServerModel(
            model_id=model_id,
            api_key=os.getenv("OPENAI_API_KEY"),
            client_kwargs={"timeout": 60, "max_retries": 2},
        )
        agent_kwargs = dict(
            model=model,
            tools=[
                DuckDuckGoSearchTool(),
                visit_webpage,
                WikipediaSearchTool(user_agent="Atlas-GAIA-agent (portfolio project)"),
                transcribe_audio,
                analyze_image,
            ],
            additional_authorized_imports=[
                "pandas", "numpy", "openpyxl", "json", "re", "math",
                "datetime", "collections", "itertools", "statistics",
            ],
            max_steps=max_steps,
        )
        try:
            self.agent = CodeAgent(**agent_kwargs, code_block_tags="markdown")
        except TypeError:  # versões antigas do smolagents não têm esse parâmetro
            self.agent = CodeAgent(**agent_kwargs)

    # ---------- anexos ----------
    def download_file(self, task_id: str, file_name: str) -> str | None:
        try:
            r = requests.get(f"{API_URL}/files/{task_id}", timeout=30)
            r.raise_for_status()
            path = os.path.join(tempfile.gettempdir(), file_name)
            with open(path, "wb") as f:
                f.write(r.content)
            return path
        except Exception as e:
            print(f"  API download failed ({e}); trying GAIA dataset...")
        try:
            return hf_hub_download(
                repo_id="gaia-benchmark/GAIA",
                repo_type="dataset",
                filename=f"2023/validation/{file_name}",
                token=os.getenv("HF_TOKEN"),
            )
        except Exception as e:
            print(f"  GAIA dataset download failed: {e}")
            return None

    # ---------- resposta ----------
    @staticmethod
    def normalize(result) -> str:
        if isinstance(result, (list, tuple)):
            answer = ", ".join(str(x) for x in result)
        else:
            answer = str(result).strip()
        return re.sub(r"(?i)^final answer:\s*", "", answer).strip()

    def __call__(self, question: str, task_id: str | None = None, file_name: str | None = None) -> str:
        prompt = INSTRUCTIONS + "\n\nQuestion: " + question
        if file_name and task_id:
            path = self.download_file(task_id, file_name)
            if path:
                prompt += (
                    f"\n\nThe question has an attached file saved at: {path}\n"
                    "Use Python to read spreadsheets/code/text files, "
                    "transcribe_audio for audio and analyze_image for images."
                )
            else:
                prompt += "\n\nThe attached file could not be downloaded. Do not invent its content."
        try:
            return self.normalize(self.agent.run(prompt))
        except Exception as e:
            return f"AGENT ERROR: {e}"


atlas = AtlasAgent()
print("Agente pronto")

Dá para ver as instruções que o smolagents manda para o modelo por baixo dos panos. É aqui que ele explica o ciclo de pensar, agir e observar:

In [ ]:
print(atlas.agent.prompt_templates["system_prompt"][:3000])

## Testando com uma pergunta

Antes de rodar tudo, testo com uma pergunta aleatória para ver se está funcionando.

In [ ]:
item = requests.get(f"{API_URL}/random-question", timeout=30).json()
print("Pergunta:", item["question"])
print("Anexo:", item.get("file_name") or "-")

answer = atlas(item["question"], item["task_id"], item.get("file_name"))
print("\nResposta do Atlas:", answer)

## Rodando todas as perguntas

São 20 perguntas. Demora uns 10 a 30 minutos.

In [ ]:
questions = requests.get(f"{API_URL}/questions", timeout=30).json()
print(f"{len(questions)} perguntas")

results = []
for i, item in enumerate(questions, 1):
    print(f"\n[{i}/{len(questions)}] {item['question'][:100]}...")
    answer = atlas(item["question"], item["task_id"], item.get("file_name"))
    print("  ->", answer)
    results.append({
        "task_id": item["task_id"],
        "question": item["question"],
        "file_name": item.get("file_name") or "",
        "submitted_answer": answer,
    })

df = pd.DataFrame(results)
df.to_csv("atlas_answers.csv", index=False)
df[["question", "submitted_answer"]]

## Enviando as respostas

Para enviar, a API do curso pede o usuário do Hugging Face, o link do código e as respostas.

Deixei `SUBMIT = False` para não enviar sem querer.

In [ ]:
SUBMIT = False
HF_USERNAME = "seu-usuario-hf"
AGENT_CODE_URL = "https://huggingface.co/spaces/seu-usuario-hf/seu-space/tree/main"

if SUBMIT:
    payload = {
        "username": HF_USERNAME,
        "agent_code": AGENT_CODE_URL,
        "answers": [{"task_id": r["task_id"], "submitted_answer": r["submitted_answer"]} for r in results],
    }
    resp = requests.post(f"{API_URL}/submit", json=payload, timeout=120)
    resp.raise_for_status()
    d = resp.json()
    print(f"Score: {d.get('score')}% ({d.get('correct_count')}/{d.get('total_attempted')} corretas)")
    print(d.get("message", ""))
else:
    print("Envio desativado (SUBMIT = False).")

## O que aprendi

Alguns problemas que apareceram enquanto eu testava:

- **A Wikipedia bloqueava o agente.** Resolvi criando minha própria ferramenta para abrir páginas, mandando um `User-Agent` na requisição.
- **Os arquivos das perguntas não baixavam pela API do curso.** Coloquei um plano B para baixar do dataset do GAIA.
- **Quando faltava o arquivo, o agente inventava a resposta.** Adicionei no prompt que ele não pode inventar dados.
- **Às vezes a resposta vinha como lista do Python** (`['a', 'b']`), o que contava como erro. Fiz uma função para arrumar o formato.

## Próximos passos

- Conseguir responder perguntas sobre vídeos do YouTube
- Testar uma versão com mais de um agente trabalhando junto
- Trocar o DuckDuckGo por uma busca mais estável